Last edited 4/24 12:00
Preprocessing + some feature engineering


In [1]:
import pandas as pd
import os
import kagglehub
from dotenv import load_dotenv

In [4]:
#Load .env file for Kaggle Credentials
load_dotenv()
username = os.getenv("KAGGLE_USERNAME")
key = os.getenv("KAGGLE_KEY")

os.environ["KAGGLE_USERNAME"] = username
os.environ["KAGGLE_KEY"] = key

In [5]:
#Locate path to the Kaggle dataset
path = kagglehub.dataset_download(
    "aiaiaidavid/the-big-dataset-of-ultra-marathon-running"
)

100%|██████████| 246M/246M [00:01<00:00, 180MB/s]

Extracting files...


In [26]:
df = pd.read_csv(f"{path}/TWO_CENTURIES_OF_UM_RACES.csv", encoding="latin1")
df = df.drop(columns=["Event dates", "Event number of finishers", "Athlete club", "Athlete average speed"])

/tmp/ipykernel_1591/1301004287.py:1: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{path}/TWO_CENTURIES_OF_UM_RACES.csv", encoding="latin1")


In [27]:
#1) Extract data since 2000s (TRY DIFFERENT THRESHOLDS)
df = df[df['Year of event'] >= 2000]

In [47]:
#2) Extract athletes that have data on a large portion of their career
# Get Athlete IDs where count exceeds 15 races (TRY DIFFERENT THRESHOLDS)
valid_athletes = df['Athlete ID'].value_counts()
valid_athletes = valid_athletes[valid_athletes > 20].index
experienced_df = df[df['Athlete ID'].isin(valid_athletes)]
experienced_df['Athlete ID'].nunique()

50969

In [48]:
#Count the number of entries for each race
experienced_df['Event name'].value_counts()

,count
Event name,
Two Oceans Marathon (RSA),49684
Comrades Marathon - Down Run (RSA),34220
Comrades Marathon - Up Run (RSA),31416
Two Oceans Marathon - 50km Split (RSA),28242
Ultra Trail Tour du Mont Blanc (UTMB) (FRA),11617
...,...
Open National Race Walking Championship Patiala (IND),1
Rohring Round the Clock 6 hour Run (USA),1
Circuito Internacional de Marcha (MEX),1


In [52]:
# Two Oceans Marathon (RSA) : 49684
df_two_oceans = experienced_df[df['Event name'] == 'Two Oceans Marathon (RSA)'].copy()
df_two_oceans.sample(10)

/tmp/ipykernel_1591/3565314029.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_two_oceans = experienced_df[df['Event name'] == 'Two Oceans Marathon (RSA)'].copy()


,Year of event,Event name,Event distance/length,Athlete performance,Athlete country,Athlete year of birth,Athlete gender,Athlete age category,Athlete ID
1895751,2019,Two Oceans Marathon (RSA),56km,5:49:12 h,RSA,1984.0,M,M35,127786
801237,2016,Two Oceans Marathon (RSA),56km,4:55:07 h,RSA,1958.0,M,M55,53166
4722895,2009,Two Oceans Marathon (RSA),56km,6:20:09 h,RSA,1957.0,F,W50,1092787
4284688,2005,Two Oceans Marathon (RSA),56km,4:57:47 h,RSA,1966.0,M,M35,259411
6359513,2015,Two Oceans Marathon (RSA),56km,6:47:06 h,RSA,1977.0,F,W35,429270
1256218,2017,Two Oceans Marathon (RSA),56km,5:54:44 h,USA,1954.0,M,M60,138069
6357985,2015,Two Oceans Marathon (RSA),56km,6:26:33 h,RSA,1957.0,M,M55,136098
4396091,2006,Two Oceans Marathon (RSA),56km,5:14:12 h,RSA,1962.0,M,M40,1075970
3870927,2000,Two Oceans Marathon (RSA),56km,4:38:29 h,RSA,1968.0,F,W23,552015
4395815,2006,Two Oceans Marathon (RSA),56km,5:04:59 h,RSA,1969.0,M,M35,1170294


In [53]:
# runners are overwhelmingly from South Africa for this race
# potential domestic/foreign feature??
df_two_oceans["Athlete country"].value_counts()

,count
Athlete country,
RSA,47406
GBR,628
GER,287
NAM,206
ZIM,156
LES,111
USA,98
SWZ,87
AUS,70


In [62]:
# seems many of the runners here have raced before
# could do some sort of time delta as a feature or just reference previous times
df_two_oceans['Athlete ID'].nunique()

6721

In [63]:
df_two_oceans.groupby('Athlete ID')['Year of event'].nunique()
#

,Year of event
Athlete ID,
110,2
148,4
313,15
340,2
405,1
...,...
1234085,1
1234899,8
1248001,1


In [57]:
#Calculate the athletes average speed in km/hr --> Used to Extract top threshold of fastest runners
#Since format H:MM:SS h harder to compare, numerically

df_two_oceans["Distance km"] = df_two_oceans["Event distance/length"].str.extract(r"(\d+\.?\d*)").astype(float)
df_two_oceans["Distance km"]
time_parts = df_two_oceans["Athlete performance"].str.replace(" h", "").str.split(":", expand=True)

df_two_oceans["hours"] = (
    time_parts[0].astype(float) +
    time_parts[1].astype(float) / 60 +
    time_parts[2].astype(float) / 3600
)

df_two_oceans["Calculated speed"] = df_two_oceans["Distance km"] / df_two_oceans["hours"]

df_two_oceans["Calculated speed"].head()

,Calculated speed
171456,17.571690
171458,17.486339
171463,17.058724
171465,16.959704
171466,16.839292


In [60]:
# adding feature for 'elite' runners, top 5% threshold can be adjusted
df_two_oceans['Elite'] = df_two_oceans['Calculated speed'] >= df_two_oceans['Calculated speed'].quantile(0.95)  # per race or per year

In [61]:
# how age impacts performance is not a linear relationship
# age category is binned, but a continuous variable would be better for the models
df_two_oceans["Age"] = df_two_oceans["Year of event"] - df_two_oceans["Athlete year of birth"]
df_two_oceans["Age sq"] = df_two_oceans["Age"]**2

age
age²
gender
year
prior performance?
1–2 interaction terms
age x gender
age x experience